In [133]:
import pandas as pd

# load the file

In [134]:
df = pd.read_csv("C:\\Users\\win11\\Downloads\\shopeasy_100k_dataset\\orders.csv")

# Check the header

In [135]:
df.head()

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8
0,O000001,C19746,P0445,02-10-2025,2,20%,Delivered,NaN,0.0
1,O000002,C19268,P0285,20-12-2025,2,10%,Returned,NaN,NaN
2,O000003,C11707,P0038,22-04-2025,3,15%,Delivered,NaN,NaN
3,O000004,C17845,P0457,27-07-2025,3,5%,Delivered,NaN,NaN
4,O000005,C10211,P0123,30-01-2025,2,15%,Delivered,NaN,NaN


# Check if there is any missing values

In [136]:
df.isnull().sum()

OrderID            0
CustomerID         0
ProductID          0
OrderDate          0
Quantity           0
Discount           0
Status             0
Unnamed: 7    100000
Unnamed: 8     99999
dtype: int64

# Check if there are any duplicate values in OrderID column

In [137]:
df["OrderID"].duplicated().sum()

np.int64(0)

# Load customer data

In [138]:
customers = pd.read_csv("C:\\Users\\win11\\Downloads\\shopeasy_100k_dataset\\customers.csv")

### Find CustomerIDs in the orders table that don't exist in the customers table, and count how many unique invalid CustomerIDs there are.

In [139]:
df[~df["CustomerID"].isin(customers["CustomerID"])]["CustomerID"].nunique()

1

# Lets see the customer id

In [140]:
df[~df["CustomerID"].isin(customers["CustomerID"])]["CustomerID"].unique()

array(['C99999'], dtype=object)

# Load product data

In [141]:
products = pd.read_csv(
    r"C:\Users\win11\Downloads\shopeasy_100k_dataset\products.csv"
)

### Find ProductIDs in the orders table that don't exist in the Products table, and count how many unique invalid CustomerIDs there are.

In [142]:
df[~df["ProductID"].isin(products["ProductID"])]["ProductID"].nunique()

1

# which productid

In [143]:
df[~df["ProductID"].isin(products["ProductID"])]["ProductID"].unique()


array(['P9999'], dtype=object)

# if there are any negative values

In [144]:
df[df["Quantity"] <=0]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8
1234,O001235,C13299,P0464,15-07-2025,-2,10%,Delivered,NaN,NaN


# A discount should normally be between 0% and 100%.


In [145]:
df["Discount"].dtype

dtype('O')

In [146]:
df["Discount"].head()

0    20%
1    10%
2    15%
3     5%
4    15%
Name: Discount, dtype: object

In [147]:
df["Discount"] = df["Discount"].str.replace("%", "").astype(float) / 100

In [148]:
df[(df["Discount"] < 0) | (df["Discount"] > 1)]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8
5678,O005679,C13590,P0477,03-08-2025,1,1.5,Delivered,NaN,NaN


# Validate OrderDate


In [149]:
df["OrderDate"].dtype


dtype('O')

In [150]:
df["OrderDate"].head()

0    02-10-2025
1    20-12-2025
2    22-04-2025
3    27-07-2025
4    30-01-2025
Name: OrderDate, dtype: object

# convert dates

In [151]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"], format="%d-%m-%Y", errors="coerce")

In [152]:
df[df["OrderDate"].isna()]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8
45678,O045679,C01908,P0153,NaT,3,0.25,Delivered,NaN,NaN


# Now let's look for inconsistent status values such as Delivered vs delivered.


In [153]:
df["Status"].unique()

array(['Delivered', 'Returned', 'Cancelled', 'delivered'], dtype=object)

In [154]:
df["Status"].value_counts()

Status
Delivered    88002
Returned      7895
Cancelled     4102
delivered        1
Name: count, dtype: int64

In [155]:
df[df["Status"] == "delivered"]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8
78901,O078902,C12578,P0011,2025-09-29,1,0.0,delivered,NaN,NaN


##### Before we start changing anything, let's do one more important check: are all orders actually within our intended analysis period, Jan–Dec 2025

In [156]:
df["OrderDate"].min(), df["OrderDate"].max()

(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

## let's deal with the invalid CustomerID.

In [157]:
df[~df["CustomerID"].isin(customers["CustomerID"])]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8
9012,O009013,C99999,P0050,2025-08-16,1,0.0,Delivered,NaN,NaN


## Invalid ProductID

In [158]:
df[~df["ProductID"].isin(products["ProductID"])]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8
34567,O034568,C07166,P9999,2025-11-24,1,0.0,Delivered,NaN,NaN


#### let's flag the invalid CustomerIDs.

In [159]:
df["InvalidCustomerID"] = ~df["CustomerID"].isin(customers["CustomerID"])

### lets flag the invalid ProductIDs

In [160]:
df["InvalidProductID"] = ~df["ProductID"].isin(products["ProductID"])

### create a flag for invalid quantities

In [161]:
df["InvalidQuantity"] = df["Quantity"] <= 0

### let's flag the invalid Discount

In [162]:
df["InvalidDiscount"] = (df["Discount"] < 0) | (df["Discount"] > 1)


### flag the invalid OrderDate

In [163]:
df["InvalidOrderDate"] = df["OrderDate"].isna()

### let's flag the inconsistent Status

In [164]:
valid_statuses = ["Delivered", "Returned", "Cancelled"]

df["InvalidStatus"] = ~df["Status"].isin(valid_statuses)

## let's see all flagged records

In [165]:
df[
    df["InvalidCustomerID"] |
    df["InvalidProductID"] |
    df["InvalidQuantity"] |
    df["InvalidDiscount"] |
    df["InvalidOrderDate"] |
    df["InvalidStatus"]
]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus
1234,O001235,C13299,P0464,2025-07-15,-2,0.10,Delivered,NaN,NaN,False,False,True,False,False,False
5678,O005679,C13590,P0477,2025-08-03,1,1.50,Delivered,NaN,NaN,False,False,False,True,False,False
9012,O009013,C99999,P0050,2025-08-16,1,0.00,Delivered,NaN,NaN,True,False,False,False,False,False
34567,O034568,C07166,P9999,2025-11-24,1,0.00,Delivered,NaN,NaN,False,True,False,False,False,False
45678,O045679,C01908,P0153,NaT,3,0.25,Delivered,NaN,NaN,False,False,False,False,True,False
78901,O078902,C12578,P0011,2025-09-29,1,0.00,delivered,NaN,NaN,False,False,False,False,False,True


### Standardize Status

In [166]:
df["Status"] = df["Status"].str.title()

In [167]:
df["Status"].unique()

array(['Delivered', 'Returned', 'Cancelled'], dtype=object)

In [168]:
df[df["Quantity"] <= 0][["OrderID", "Quantity", "Status"]]


,OrderID,Quantity,Status
1234,O001235,-2,Delivered


## Create a cleaned dataframe

In [169]:
clean_df = df.copy()

In [170]:
clean_df = clean_df[clean_df["Quantity"] > 0]

In [171]:
clean_df[clean_df["Quantity"] <= 0]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus


### Clean the invalid Discount

In [172]:
clean_df = clean_df[clean_df["Discount"].between(0, 1)]

In [173]:
clean_df[~clean_df["Discount"].between(0, 1)]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus


In [174]:
len(df), len(clean_df)

(100000, 99998)

In [175]:
df[df["Quantity"] <= 0][["OrderID", "Quantity", "Status"]]


,OrderID,Quantity,Status
1234,O001235,-2,Delivered


In [176]:
set(df["OrderID"]) - set(clean_df["OrderID"])

{'O001235', 'O005679'}

In [177]:
df[df["OrderID"] == "O005679"]


,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus
5678,O005679,C13590,P0477,2025-08-03,1,1.5,Delivered,NaN,NaN,False,False,False,True,False,False


In [178]:
clean_df[clean_df["Discount"] > 1]
len(clean_df)

99998

### Check the invalid customer record

In [179]:
clean_df[clean_df["InvalidCustomerID"] == True]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus
9012,O009013,C99999,P0050,2025-08-16,1,0.0,Delivered,NaN,NaN,True,False,False,False,False,False


### Remove this invalid customer record

In [180]:
clean_df = clean_df[clean_df["InvalidCustomerID"] == False]

In [181]:
clean_df[clean_df["InvalidCustomerID"] == True]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus


In [182]:
len(clean_df)

99997

### Remove this invalid Product record

In [183]:
clean_df[clean_df["InvalidProductID"] == True]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus
34567,O034568,C07166,P9999,2025-11-24,1,0.0,Delivered,NaN,NaN,False,True,False,False,False,False


In [184]:
clean_df = clean_df[clean_df["InvalidProductID"] == False]

In [185]:
clean_df[clean_df["InvalidProductID"] == True]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus


In [186]:
len(clean_df)

99996

### Remove the invalid OrderDate record

In [187]:
clean_df[clean_df["InvalidOrderDate"] == True]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus
45678,O045679,C01908,P0153,NaT,3,0.25,Delivered,NaN,NaN,False,False,False,False,True,False


In [188]:
clean_df = clean_df[clean_df["InvalidOrderDate"] == False]

In [189]:
clean_df[clean_df["InvalidOrderDate"] == True]

,OrderID,CustomerID,ProductID,OrderDate,Quantity,Discount,Status,Unnamed: 7,Unnamed: 8,InvalidCustomerID,InvalidProductID,InvalidQuantity,InvalidDiscount,InvalidOrderDate,InvalidStatus


In [190]:
len(clean_df)

99995

In [191]:
clean_df[clean_df["Quantity"] <= 0]
clean_df[(clean_df["Discount"] < 0) | (clean_df["Discount"] > 1)]
clean_df[~clean_df["CustomerID"].isin(customers["CustomerID"])]
clean_df[~clean_df["ProductID"].isin(products["ProductID"])]
clean_df[clean_df["OrderDate"].isna()]
len(clean_df)

99995

In [192]:
clean_df["Status"].value_counts()

Status
Delivered    87998
Returned      7895
Cancelled     4102
Name: count, dtype: int64

In [193]:
clean_df.isnull().sum()

OrderID                  0
CustomerID               0
ProductID                0
OrderDate                0
Quantity                 0
Discount                 0
Status                   0
Unnamed: 7           99995
Unnamed: 8           99994
InvalidCustomerID        0
InvalidProductID         0
InvalidQuantity          0
InvalidDiscount          0
InvalidOrderDate         0
InvalidStatus            0
dtype: int64

In [194]:
flag_columns = [
    "InvalidCustomerID",
    "InvalidProductID",
    "InvalidQuantity",
    "InvalidDiscount",
    "InvalidOrderDate",
    "InvalidStatus"
]

clean_df = clean_df.drop(columns=flag_columns)

In [195]:
clean_df.columns

Index(['OrderID', 'CustomerID', 'ProductID', 'OrderDate', 'Quantity',
       'Discount', 'Status', 'Unnamed: 7', 'Unnamed: 8'],
      dtype='object')

In [196]:
clean_df.to_csv(
    r"C:\Users\win11\Downloads\shopeasy_100k_dataset\clean_orders.csv",
    index=False
)

In [197]:
import os

os.path.exists(
    r"C:\Users\win11\Downloads\shopeasy_100k_dataset\clean_orders.csv"
)

True